# 12 — Verified 9×9 teacher data and typed targets

Prepare a fixed external teaching corpus for the AHR Scout and Worker. Every accepted game is replayed from empty with **our no-suicide positional-superko rules**. Features and legal masks are recomputed from causal history. Teacher policy/value/score remain external estimates; the played action is never substituted for a policy target. Whole games with incompatible actions or board histories are rejected.

**Inputs:** immutable `quintic/go9x9` release `final-9x9-v1`, commit `4f579b0a21c456f3bb568174d384056351124cd5`; tagged definitions from 00–02. **Outputs:** checksum-verified persistent raw tar shards, bounded `.npy` feature/target shards, strict manifest, audit and preparation cursor. **Runtime:** Run All performs a small real-data check; full preparation can take tens of minutes, depending on selected size and CPU throughput. It performs no model training. **Storage:** `data/ahr_v2_teacher/`; preparation recovery in `checkpoints/ahr-v2-teacher-data-<cache_id>/`.

**Resume:** rerun `go2_prepare_teacher_cache(PROJECT, cache_id, train_positions, dev_positions, ...)` with the identical arguments. A single writer lock protects the cursor. Each committed shard is hashed before its cursor is atomically advanced. Configuration, source or input mismatches stop the run. Interrupted uncommitted shard directories are removed only after verifying that the cursor never referenced them. The latest two valid preparation cursors are retained; published cache manifests are immutable once complete.

The dataset allows multi-stone suicide and uses native pass-alive terminal scoring. We never copy its legal masks or native final scores into our rule engine. Any observed outcome/ownership auxiliary target is recomputed from our fully replayed terminal state and enabled only when its score agrees with the publisher's score. Test games are never opened or consumed.

Sources: [dataset card](https://huggingface.co/datasets/quintic/go9x9), [pinned release](https://huggingface.co/datasets/quintic/go9x9/tree/4f579b0a21c456f3bb568174d384056351124cd5). License declaration: GPL-3.0; teacher checkpoint upstream terms remain recorded in `contract.json`.

## Load the frozen rule and feature definitions

In [ ]:
from pathlib import Path
import nbformat, json, time, os
PROJECT=Path.cwd()
while not (PROJECT/'AGENTS.md').exists() and PROJECT.parent!=PROJECT: PROJECT=PROJECT.parent
if not (PROJECT/'AGENTS.md').exists(): raise RuntimeError('Open within the AHR Go workspace.')
for cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]): exec(compile(cell.source,'00 ahr-export','exec'),globals())
for filename in ['01_rules_and_replay.ipynb','02_scout_worker_network.ipynb']:
    go2_import_exports(PROJECT/GO2_ROOT/filename,globals())
print('Frozen rules and 21-plane features loaded; no training cells imported.')

## Pin, resume and verify the source download
Partial HTTP transfers stay on the persistent disk. Every completed source file and each selected game payload is checked against the immutable release.

In [ ]:
import requests, hashlib, io, gzip, shutil, concurrent.futures, fcntl, contextlib
GO2_TEACHER_NOTEBOOK='12_teacher_dataset_and_targets.ipynb'
GO2_TEACHER_COMMIT='4f579b0a21c456f3bb568174d384056351124cd5'
GO2_TEACHER_RELEASE='6a26ffa743f3c7be95d783102f6b3596e0ce69893c5a8e28dcca154dc54f05f4'
GO2_TEACHER_SUMS_SHA='05910a9908a23657cce222a1d41b504fab85c5fdc6c10a46d181275e6cf86d28'
GO2_TEACHER_HASH='a1298ce1adc1dad7bd868ca962b2384cc8388ed373a00e6bae1114fa6f9e2d61'
GO2_TEACHER_BASE=f'https://huggingface.co/datasets/quintic/go9x9/resolve/{GO2_TEACHER_COMMIT}/'

def go2_teacher_source_identity(project):
    project=Path(project)
    return {name:go2_code_sha(project/GO2_ROOT/name) for name in [
        '00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb',
        '02_scout_worker_network.ipynb',GO2_TEACHER_NOTEBOOK]}

@contextlib.contextmanager
def go2_teacher_writer_lock(path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('a+b') as handle:
        try:fcntl.flock(handle.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:raise RuntimeError('Another preparation writer owns this cache.')
        try:yield
        finally:fcntl.flock(handle.fileno(),fcntl.LOCK_UN)

def go2_teacher_download_file(root,name,expected_sha=None):
    root=Path(root);target=root/name
    if target.exists():
        if expected_sha is not None and go2_file_sha(target)!=expected_sha:
            raise ValueError('Existing source artifact checksum mismatch: '+name)
        return target
    target.parent.mkdir(parents=True,exist_ok=True)
    partial=target.with_name(target.name+'.partial')
    for attempt in range(6):
        offset=partial.stat().st_size if partial.exists() else 0
        headers={'Range':f'bytes={offset}-'} if offset else {}
        try:
            with requests.get(GO2_TEACHER_BASE+name,headers=headers,stream=True,timeout=(20,30)) as response:
                if response.status_code==416 and offset:
                    # Server confirms end-of-file; accept only the pinned checksum.
                    if expected_sha and go2_file_sha(partial)==expected_sha:break
                    raise ValueError('Partial source has invalid length or bytes: '+name)
                response.raise_for_status()
                mode='ab' if offset and response.status_code==206 else 'wb'
                if mode=='ab' and not response.headers.get('Content-Range','').startswith(f'bytes {offset}-'):
                    raise ValueError('Range response did not start at the persisted byte cursor.')
                with partial.open(mode) as handle:
                    for chunk in response.iter_content(4*1024*1024):
                        if chunk:handle.write(chunk)
                    handle.flush();os.fsync(handle.fileno())
            break
        except (requests.RequestException,OSError):
            if attempt==5:raise
            time.sleep(min(1+attempt,5))
    if expected_sha is not None and go2_file_sha(partial)!=expected_sha:
        raise ValueError('Downloaded source artifact checksum mismatch: '+name)
    os.replace(partial,target)
    return target

def go2_download_teacher_source(project=None,full=True,workers=4):
    """Persist source release without extracting games or downloading executable helpers."""
    project=Path(project or PROJECT);root=project/'data/ahr_v2_teacher/raw'/GO2_TEACHER_COMMIT
    root.mkdir(parents=True,exist_ok=True)
    with go2_teacher_writer_lock(root/'.download.lock'):
        sums=go2_teacher_download_file(root,'SHA256SUMS',GO2_TEACHER_SUMS_SHA)
        checks={line.split(None,1)[1].strip():line.split(None,1)[0] for line in sums.read_text().splitlines() if line.strip()}
        for name in ['README.md','manifest.json','contract.json']:
            go2_teacher_download_file(root,name,checks[name])
        manifest=json.loads((root/'manifest.json').read_text());contract=json.loads((root/'contract.json').read_text())
        if manifest['release_id']!=GO2_TEACHER_RELEASE or contract['teacher']['sha256']!=GO2_TEACHER_HASH:
            raise ValueError('Pinned release/teacher identity disagrees with source metadata.')
        names=[x['path'] for x in manifest['indices']]
        names += [x['path'] for x in (manifest['shards'] if full else manifest['shards'][:1])]
        pending_bytes=sum(x['bytes'] for x in manifest['indices']+manifest['shards'] if x['path'] in names and not (root/x['path']).exists())
        if shutil.disk_usage(root).free<pending_bytes+2*1024**3:raise RuntimeError('Insufficient persistent storage for source download.')
        with concurrent.futures.ThreadPoolExecutor(max_workers=int(workers)) as pool:
            results=list(pool.map(lambda name:go2_teacher_download_file(root,name,checks[name]),names))
        inventory={name:{'sha256':checks[name],'bytes':(root/name).stat().st_size} for name in ['README.md','manifest.json','contract.json']+names}
        report={'schema':'ahr_teacher_source_v1','dataset':'quintic/go9x9','commit':GO2_TEACHER_COMMIT,
            'release_id':GO2_TEACHER_RELEASE,'teacher_sha256':GO2_TEACHER_HASH,'checksum_list_sha256':GO2_TEACHER_SUMS_SHA,
            'files':inventory,'complete_payload':full,'persistent_path':str(root),'verified_utc':go2_now()}
        go2_atomic_json(root/'verified_source.json',report)
    return {'root':root,'manifest':manifest,'contract':contract,'inventory':report}

def go2_teacher_index(project=None):
    source=go2_download_teacher_source(project,full=True)
    records=[];seen=set();families={}
    for info in source['manifest']['indices']:
        with gzip.open(source['root']/info['path'],'rt') as handle:
            for line in handle:
                row=json.loads(line)
                gid=row['game_id'];family=row['opening_family'];split=row['split']
                if gid in seen:raise ValueError('Duplicate source game ID.')
                seen.add(gid)
                if family in families and families[family]!=split:raise ValueError('Opening family crosses split.')
                families[family]=split
                bucket=int(family[:8],16)%100
                expected='test' if bucket<5 else 'validation' if bucket<10 else 'train'
                if split!=expected:raise ValueError('Source split bucket mismatch.')
                if split!='test':records.append(row)
    return source,records

def go2_teacher_read_game(source_root,row):
    path=Path(source_root)/row['shard']
    with path.open('rb') as handle:
        handle.seek(row['offset']);payload=handle.read(row['bytes'])
    if len(payload)!=row['bytes'] or hashlib.sha256(payload).hexdigest()!=row['sha256']:
        raise ValueError('Game payload checksum mismatch: '+row['game_id'])
    with np.load(io.BytesIO(payload),allow_pickle=False) as archive:
        metadata=json.loads(archive['metadata'].tobytes())
        arrays={name:archive[name] for name in archive.files if name!='metadata'}
    for key in ['game_id','opening_family','split','opponent_index']:
        if metadata.get(key)!=row.get(key):raise ValueError('Index/payload metadata mismatch: '+key)
    if metadata.get('contract_id')!=source_root_contract_id(source_root):raise ValueError('Game contract ID mismatch.')
    return metadata,arrays

@lru_cache(maxsize=4)
def source_root_contract_id(root):
    return json.loads((Path(root)/'manifest.json').read_text())['contract_id']


## Recompute history, legality and target provenance
Raw teacher policy is renormalized only over our legal actions. Raw signed teacher value and available normalized score retain the current-player perspective. Observed terminal auxiliaries have separate validity masks; UNKNOWN and absent fields never become labels.

In [ ]:
GO2_TEACHER_FIELDS={
    'features':(np.float16,(21,9,9)),'legal':(np.bool_,(82,)),'policy':(np.float32,(82,)),
    'value':(np.float32,()),'score':(np.float32,()),'score_valid':(np.bool_,()),
    'outcome':(np.float32,(3,)),'outcome_valid':(np.bool_,()),
    'ownership':(np.int8,(9,9)),'ownership_valid':(np.bool_,()),
    'game_id':('S64',()),'family':('S64',()),'ply':(np.int16,()),'opponent':(np.int8,()),
    'teacher_turn':(np.bool_,()),'removed_policy_mass':(np.float32,())}

def go2_teacher_opening_family(actions):
    variants=[tuple(go2_transform_action(int(a),9,k) for a in actions[:8]) for k in range(8)]
    return hashlib.sha256(json.dumps(min(variants),separators=(',',':')).encode()).hexdigest()

def go2_teacher_validate_arrays(metadata,arrays):
    length=int(metadata['rows'])
    schema={'stones':((length,9,9),'u'),'legal':((length,82),'b'),
        'raw_policy':((length,82),'f'),'raw_value':((length,),'f'),
        'search_policy':((length,82),'f'),'search_value':((length,),'f'),
        'search_policy_valid':((length,),'b'),'root_edge_visits':((length,82),'i'),
        'raw_score':((length,),'f'),'raw_score_valid':((length,),'b'),
        'teacher_visits':((length,),'i'),'actions':((length,),'i')}
    if set(arrays)!=set(schema):raise ValueError('Unexpected source NPZ schema.')
    for name,(shape,kind) in schema.items():
        value=arrays[name]
        if value.shape!=shape or value.dtype.kind!=kind:raise ValueError('Source field shape/type mismatch: '+name)
    if not length or not metadata['terminal'] or metadata['teacher_sha256']!=GO2_TEACHER_HASH:
        raise ValueError('Invalid terminal/teacher metadata.')
    if not np.isin(arrays['stones'],[0,1,2]).all() or np.any((arrays['actions']<0)|(arrays['actions']>81)):
        raise ValueError('Invalid source board/action coordinates.')
    if go2_teacher_opening_family(arrays['actions'])!=metadata['opening_family']:
        raise ValueError('Full causal opening family mismatch.')
    raw=arrays['raw_policy'];legal=arrays['legal']
    if not np.isfinite(raw).all() or (raw<0).any() or (raw[~legal]!=0).any() or not np.allclose(raw.sum(1),1,atol=1e-5,rtol=0):
        raise ValueError('Invalid teacher raw policy/support.')
    values=arrays['raw_value']
    if not np.isfinite(values).all() or (np.abs(values)>1+1e-6).any():raise ValueError('Invalid teacher signed value.')
    valid=arrays['raw_score_valid']
    if not np.isfinite(arrays['raw_score'][valid]).all():raise ValueError('Invalid available teacher score.')
    if (arrays['root_edge_visits']<0).any():raise ValueError('Negative teacher edge visit count.')
    teacher_turn=np.array([(1 if t%2==0 else 2)==metadata['expert_color'] for t in range(length)])
    if (arrays['search_policy_valid']&~teacher_turn).any():raise ValueError('Teacher search target marked valid on opponent turn.')
    return length

def go2_teacher_convert_game(source_root,row,max_positions=32):
    """All-or-nothing exact replay, then selected causal features and typed targets."""
    metadata,arrays=go2_teacher_read_game(Path(source_root),row)
    length=go2_teacher_validate_arrays(metadata,arrays)
    states=[];state=go2_initial_state(9,7.5)
    try:
        for t,action in enumerate(arrays['actions']):
            source_board=tuple(int(v) if v!=2 else -1 for v in arrays['stones'][t].reshape(-1))
            if state.terminal or state.board!=source_board:
                return {'accepted':False,'game_id':row['game_id'],'reason':'board_or_terminal_history_incompatible','ply':t}
            states.append(state)
            state=go2_step(state,int(action))
    except Go2IllegalMove as error:
        return {'accepted':False,'game_id':row['game_id'],'reason':'illegal_under_our_rules','ply':len(states)-1,'detail':str(error)}
    if not state.terminal:
        return {'accepted':False,'game_id':row['game_id'],'reason':'not_terminal_under_our_rules'}
    selected=np.unique(np.linspace(0,length-1,min(int(max_positions),length),dtype=np.int64))
    result={key:np.empty((len(selected),)+shape,dtype=dtype) for key,(dtype,shape) in GO2_TEACHER_FIELDS.items()}
    final=go2_score(state);winner=1 if final['score_black']>0 else -1 if final['score_black']<0 else 0
    observed_valid=bool(np.isfinite(metadata['white_score']) and abs(float(metadata['white_score'])+final['score_black'])<=1e-5)
    for index,t in enumerate(selected):
        position=states[int(t)]
        features=go2_encode_state(position)
        legal=np.asarray(go2_legal_mask(position),dtype=bool)
        if not np.array_equal(features[16].astype(bool).reshape(-1),legal[:-1]):raise AssertionError('Feature/target legal-mask disagreement.')
        policy=arrays['raw_policy'][t].astype(np.float64)*legal
        mass=float(policy.sum())
        if mass<=1e-12:return {'accepted':False,'game_id':row['game_id'],'reason':'no_compatible_teacher_policy_mass','ply':int(t)}
        policy/=mass
        teacher_score=float(arrays['raw_score'][t])
        score_valid=bool(arrays['raw_score_valid'][t] and np.isfinite(teacher_score) and abs(teacher_score)<=88.5)
        value=position.to_play*winner
        result['features'][index]=features
        result['legal'][index]=legal;result['policy'][index]=policy
        result['value'][index]=arrays['raw_value'][t]
        result['score'][index]=teacher_score/88.5 if score_valid else 0
        result['score_valid'][index]=score_valid
        result['outcome'][index]=[value==-1,value==0,value==1]
        result['outcome_valid'][index]=observed_valid
        result['ownership'][index]=np.array(final['ownership_black'],dtype=np.int8).reshape(9,9)*position.to_play
        result['ownership_valid'][index]=observed_valid
        result['game_id'][index]=row['game_id'].encode();result['family'][index]=row['opening_family'].encode()
        result['ply'][index]=t;result['opponent'][index]=row['opponent_index']
        result['teacher_turn'][index]=(1 if position.to_play==1 else 2)==metadata['expert_color']
        result['removed_policy_mass'][index]=max(0,1-mass)
    return {'accepted':True,'arrays':result,'game_id':row['game_id'],'family':row['opening_family'],
        'split':row['split'],'rows':len(selected),'source_game_rows':length,'observed_auxiliary_valid':observed_valid,
        'our_terminal_score_black':final['score_black'],'publisher_terminal_score_white':metadata['white_score']}

def go2_teacher_balanced_order(records,split,seed=2026100812):
    buckets=[[] for _ in range(8)]
    for row in records:
        if row['split']==split:buckets[int(row['opponent_index'])].append(row)
    for bucket in buckets:bucket.sort(key=lambda row:hashlib.sha256((str(seed)+row['game_id']).encode()).digest())
    ordered=[]
    for index in range(max(map(len,buckets))):
        for bucket in buckets:
            if index<len(bucket):ordered.append(bucket[index])
    return ordered


## Commit bounded feature shards and exact preparation cursors
Equal accepted-position quotas prevent the raw corpus opponent imbalance from determining the training mixture. At most 32 positions span each complete game. Source test families stay unopened. Shards are uncompressed `.npy` data, suitable for memory mapping.

In [ ]:
def go2_teacher_cache_path(project,cache_id):
    if not isinstance(cache_id,str) or not cache_id or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_' for c in cache_id):
        raise ValueError('cache_id must be a simple path-safe identifier.')
    return Path(project)/'data/ahr_v2_teacher/prepared'/cache_id

def go2_teacher_save_cursor(checkpoint_root,cursor):
    root=Path(checkpoint_root);root.mkdir(parents=True,exist_ok=True)
    path=root/f"cursor-{cursor['commits']:08d}.json"
    go2_atomic_json(path,cursor)
    old=json.loads((root/'manifest.json').read_text()) if (root/'manifest.json').exists() else {'checkpoints':[]}
    entries=[{'path':path.name,'sha256':go2_file_sha(path),'commit':cursor['commits']}]+old['checkpoints']
    entries=entries[:2]
    go2_atomic_json(root/'manifest.json',{'schema':'ahr_teacher_producer_checkpoints_v1','latest':entries[0],
        'checkpoints':entries,'resume_notebook':str(Path(GO2_ROOT)/GO2_TEACHER_NOTEBOOK),
        'resume_function':'go2_prepare_teacher_cache with the identical cache_id/configuration',
        'identity':cursor['identity']})
    keep={e['path'] for e in entries}
    for previous in root.glob('cursor-*.json'):
        if previous.name not in keep:previous.unlink()

def go2_teacher_verify_shards(cache_root,shards):
    root=Path(cache_root)
    for shard in shards:
        games=shard.get('games_artifact')
        if games:
            game_path=root/games['path']
            if not game_path.exists() or game_path.stat().st_size!=games['bytes'] or go2_file_sha(game_path)!=games['sha256']:
                raise ValueError('Prepared game provenance checksum mismatch.')
        for field,artifact in shard['files'].items():
            path=root/artifact['path']
            if not path.exists() or path.stat().st_size!=artifact['bytes'] or go2_file_sha(path)!=artifact['sha256']:
                raise ValueError('Prepared feature/target shard checksum mismatch: '+str(path))
            array=np.load(path,mmap_mode='r',allow_pickle=False)
            dtype,shape=GO2_TEACHER_FIELDS[field]
            if array.shape!=(shard['rows'],)+shape or array.dtype!=np.dtype(dtype):
                raise ValueError('Prepared shard array shape/type mismatch: '+field)
    return True

def go2_prepare_teacher_cache(project=None,cache_id='go9x9-teacher-v1',train_positions=500000,
        dev_positions=8192,max_positions_per_game=32,games_per_commit=128,seed=2026100812,
        max_commits=None,on_commit=None):
    """Prepare a deterministic, bounded, restart-safe balanced view on persistent disk.

    Complete games are validated before any selected rows are committed. Official
    test payloads are never opened. This preparation uses no random generator:
    ordering is an immutable SHA256 rank, and rows evenly span each legal game.
    """
    project=Path(project or PROJECT);cache_root=go2_teacher_cache_path(project,cache_id)
    if not isinstance(train_positions,int) or not isinstance(dev_positions,int) or min(train_positions,dev_positions)<8:
        raise ValueError('Train and dev counts must be integers >=8.')
    if max_positions_per_game<1 or games_per_commit<1:raise ValueError('Invalid preparation chunk limits.')
    config={'schema':'ahr_teacher_cache_recipe_v1','cache_id':cache_id,'train_positions':train_positions,
        'dev_positions':dev_positions,'max_positions_per_game':int(max_positions_per_game),
        'games_per_commit':int(games_per_commit),'seed':int(seed),'rule_profile':GO2_RULE_PROFILE,
        'feature_schema':'go2_21planes_v1','policy':'raw_teacher_masked_renormalized',
        'value':'raw_signed_teacher_current_player','score':'raw_teacher_if_available_within88.5',
        'observed_auxiliary':'our_complete_terminal_replay_if_publisher_score_agrees',
        'balance':'equal_accepted_positions_per_opponent','test_consumed':False,
        'commit':GO2_TEACHER_COMMIT,'release_id':GO2_TEACHER_RELEASE,'teacher_sha256':GO2_TEACHER_HASH,
        'source_identity':go2_teacher_source_identity(project)}
    identity=go2_digest(config);cache_root.mkdir(parents=True,exist_ok=True)
    checkpoint_root=project/'checkpoints'/('ahr-v2-teacher-data-'+cache_id)
    with go2_teacher_writer_lock(checkpoint_root/'.writer.lock'):
        if (cache_root/'manifest.json').exists():
            complete=json.loads((cache_root/'manifest.json').read_text())
            if complete['preparation_identity']!=identity:raise ValueError('Completed cache configuration/source mismatch.')
            return go2_teacher_dataset(project,cache_id,verify=True)
        if shutil.disk_usage(cache_root).free<(train_positions+dev_positions)*4200+2*1024**3:
            raise RuntimeError('Insufficient persistent storage for the requested prepared bank.')
        source,records=go2_teacher_index(project)
        ordered={split:go2_teacher_balanced_order(records,'train' if split=='train' else 'validation',seed)
                 for split in ['train','dev']}
        if (checkpoint_root/'manifest.json').exists():
            saved=json.loads((checkpoint_root/'manifest.json').read_text())
            if saved['identity']!=identity:raise ValueError('Preparation source/configuration identity mismatch; refuse restart.')
            last=checkpoint_root/saved['latest']['path']
            if not last.exists() or go2_file_sha(last)!=saved['latest']['sha256']:
                raise ValueError('Latest preparation cursor checksum mismatch; refuse silent restart.')
            cursor=json.loads(last.read_text())
            if cursor['identity']!=identity or cursor['config']!=config:raise ValueError('Preparation cursor identity mismatch.')
            go2_teacher_verify_shards(cache_root,cursor['shards'])
        else:
            cursor={'schema':'ahr_teacher_producer_cursor_v1','identity':identity,'config':config,
                'commits':0,'source_cursors':{'train':0,'dev':0},'counts':{'train':[0]*8,'dev':[0]*8},
                'accepted_games':{'train':0,'dev':0},'rejected':{},'shards':[],
                'families':{'train':[],'dev':[]},'rng':{'algorithm':'deterministic_sha256_rank_no_rng'},
                'inflight_game':None,'status':'PREPARING','created_utc':go2_now()}
            go2_teacher_save_cursor(checkpoint_root,cursor)
        # A directory absent from the committed cursor can only be an incomplete
        # write from this producer. Never remove a cursor-referenced shard.
        referenced={s['directory'] for s in cursor['shards']}
        for stale in cache_root.glob('shard-*'):
            if stale.name not in referenced:
                if not stale.is_dir():raise ValueError('Unexpected uncommitted shard artifact.')
                shutil.rmtree(stale)
        targets={split:[n//8+(i<n%8) for i in range(8)] for split,n in [('train',train_positions),('dev',dev_positions)]}
        started=time.monotonic();initial_rows=sum(sum(v) for v in cursor['counts'].values());committed_now=0
        while any(cursor['counts'][split]!=targets[split] for split in ['train','dev']):
            split=next(s for s in ['train','dev'] if cursor['counts'][s]!=targets[s])
            batches=[];metadata=[];attempts=0;chunk_started=time.monotonic()
            while attempts<games_per_commit and time.monotonic()-chunk_started<100:
                if cursor['counts'][split]==targets[split]:break
                pos=cursor['source_cursors'][split]
                if pos>=len(ordered[split]):raise RuntimeError('Insufficient compatible games to fill requested balanced bank.')
                row=ordered[split][pos];cursor['source_cursors'][split]+=1
                opponent=int(row['opponent_index'])
                remaining=targets[split][opponent]-cursor['counts'][split][opponent]
                if remaining<=0:continue
                attempts+=1
                result=go2_teacher_convert_game(source['root'],row,max_positions_per_game)
                if not result['accepted']:
                    reason=result['reason'];cursor['rejected'][reason]=cursor['rejected'].get(reason,0)+1
                    continue
                take=min(remaining,result['rows'])
                batches.append({key:value[:take] for key,value in result['arrays'].items()})
                metadata.append({key:value for key,value in result.items() if key!='arrays'})
                cursor['counts'][split][opponent]+=take;cursor['accepted_games'][split]+=1
                cursor['families'][split].append(result['family'])
            cursor['commits']+=1
            if batches:
                directory=f"shard-{cursor['commits']:08d}-{split}";folder=cache_root/directory;folder.mkdir()
                files={};rows_count=sum(len(batch['value']) for batch in batches)
                for field in GO2_TEACHER_FIELDS:
                    path=folder/(field+'.npy');array=np.concatenate([batch[field] for batch in batches],axis=0)
                    with path.open('wb') as handle:
                        np.save(handle,array,allow_pickle=False);handle.flush();os.fsync(handle.fileno())
                    files[field]={'path':str(path.relative_to(cache_root)),'sha256':go2_file_sha(path),'bytes':path.stat().st_size}
                go2_atomic_json(folder/'games.json',metadata)
                games_path=folder/'games.json'
                cursor['shards'].append({'split':split,'directory':directory,'rows':rows_count,'files':files,
                    'games_artifact':{'path':str(games_path.relative_to(cache_root)),'sha256':go2_file_sha(games_path),'bytes':games_path.stat().st_size}})
            cursor['updated_utc']=go2_now()
            cursor['elapsed_seconds']=time.monotonic()-started
            cursor['new_rows_per_second']=(sum(sum(v) for v in cursor['counts'].values())-initial_rows)/max(.001,cursor['elapsed_seconds'])
            go2_teacher_save_cursor(checkpoint_root,cursor);committed_now+=1
            status={k:cursor[k] for k in ['identity','commits','counts','accepted_games','rejected','source_cursors','status','updated_utc','new_rows_per_second']}
            go2_atomic_json(cache_root/'preparation_status.json',status)
            if on_commit is not None:on_commit(status)
            if max_commits is not None and committed_now>=max_commits:
                return {'status':'PREPARING','cache_path':str(cache_root),'preparation_identity':identity,'cursor':cursor}
        if set(cursor['families']['train'])&set(cursor['families']['dev']):raise AssertionError('Prepared opening family leak.')
        go2_teacher_verify_shards(cache_root,cursor['shards'])
        manifest={'schema':'ahr_teacher_cache_v1','status':'COMPLETE','preparation_identity':identity,'config':config,
            'source_inventory_sha256':go2_digest(source['inventory']['files']),
            'source_release_id':GO2_TEACHER_RELEASE,'teacher_sha256':GO2_TEACHER_HASH,
            'counts':{split:sum(cursor['counts'][split]) for split in ['train','dev']},
            'opponent_counts':cursor['counts'],'accepted_games':cursor['accepted_games'],'rejected':cursor['rejected'],
            'source_cursors':cursor['source_cursors'],'families':{k:sorted(set(v)) for k,v in cursor['families'].items()},
            'shards':cursor['shards'],'target_provenance':{
                'policy':{'certainty':'teacher_estimate','perspective':'action','source_field':'raw_policy','legal_renormalized':True},
                'value':{'certainty':'teacher_estimate','perspective':'current_player','source_field':'raw_value','draw_mass_unknown':True},
                'score':{'certainty':'teacher_estimate','perspective':'current_player','normalization':88.5,'mask':'score_valid'},
                'outcome':{'certainty':'terminal_verified','perspective':'current_player','mask':'outcome_valid','source':'our_replay_score_agreement_only'},
                'ownership':{'certainty':'terminal_verified','perspective':'current_player','mask':'ownership_valid','source':'our_replay_score_agreement_only'}},
            'created_utc':go2_now(),'test_consumed':False}
        manifest['identity']=go2_digest(manifest)
        go2_atomic_json(cache_root/'manifest.json',manifest)
        cursor['status']='COMPLETE';go2_teacher_save_cursor(checkpoint_root,cursor)
    return go2_teacher_dataset(project,cache_id,verify=True)


## Read batches without loading the full bank
The public API returns tensors with explicit head masks. Memory mapping and a bounded 32-shard mapping cache support random batch indices. Notebook 13 owns sampling RNG, D4 augmentation and training checkpoints.

In [ ]:
from collections import OrderedDict

def go2_teacher_dataset(project=None,cache_id='go9x9-teacher-v1',verify=True):
    """Load immutable metadata and bounded mmap cache; no bulk feature allocation."""
    project=Path(project or PROJECT);root=go2_teacher_cache_path(project,cache_id)
    if not (root/'manifest.json').exists():raise FileNotFoundError('Prepared cache is not complete: '+str(root))
    manifest=json.loads((root/'manifest.json').read_text())
    signed=dict(manifest);identity=signed.pop('identity',None)
    if not identity or go2_digest(signed)!=identity:raise ValueError('Prepared manifest identity mismatch.')
    if manifest.get('schema')!='ahr_teacher_cache_v1' or manifest.get('status')!='COMPLETE' or manifest.get('test_consumed') is not False:
        raise ValueError('Prepared cache schema/status/test-split contract mismatch.')
    config=manifest['config']
    if config['source_identity']!=go2_teacher_source_identity(project) or config['commit']!=GO2_TEACHER_COMMIT:
        raise ValueError('Prepared cache/source mismatch; refuse silently using changed feature code.')
    if manifest['preparation_identity']!=go2_digest(config):raise ValueError('Prepared recipe identity mismatch.')
    if set(manifest['families']['train'])&set(manifest['families']['dev']):raise ValueError('Prepared family leakage.')
    if verify:go2_teacher_verify_shards(root,manifest['shards'])
    dataset={'manifest':manifest,'identity':identity,'cache_path':str(root),'status':'COMPLETE','_mmap_cache':OrderedDict()}
    for split in ['train','dev']:
        shards=[s for s in manifest['shards'] if s['split']==split]
        ends=np.cumsum([s['rows'] for s in shards],dtype=np.int64)
        count=int(ends[-1]) if len(ends) else 0
        if count!=manifest['counts'][split]:raise ValueError('Prepared split row count mismatch.')
        dataset[split]={'count':count,'row_count':count,'shards':shards,'ends':ends,
            'family_count':len(manifest['families'][split]),'game_count':manifest['accepted_games'][split]}
    return dataset

go2_teacher_open_cache=go2_teacher_dataset

def go2_teacher_sample_batch(dataset,indices,device='cpu',split='train'):
    if split not in ['train','dev']:raise ValueError('Only train/dev prepared views are available; test remains unopened.')
    if isinstance(indices,torch.Tensor):indices=indices.detach().cpu().numpy()
    indices=np.asarray(indices)
    if indices.ndim!=1 or indices.dtype.kind not in 'iu' or not len(indices):raise ValueError('Batch indices must be a nonempty integer vector.')
    view=dataset[split]
    if (indices<0).any() or (indices>=view['count']).any():raise IndexError('Prepared batch index outside split bounds.')
    required=['features','legal','policy','value','score','score_valid','outcome','outcome_valid','ownership','ownership_valid']
    buffers={key:np.empty((len(indices),)+GO2_TEACHER_FIELDS[key][1],dtype=GO2_TEACHER_FIELDS[key][0]) for key in required}
    shard_ids=np.searchsorted(view['ends'],indices,side='right')
    cache=dataset['_mmap_cache'];root=Path(dataset['cache_path'])
    for shard_id in np.unique(shard_ids):
        shard=view['shards'][int(shard_id)];key=(split,int(shard_id))
        if key not in cache:
            cache[key]={name:np.load(root/shard['files'][name]['path'],mmap_mode='r',allow_pickle=False) for name in required}
        mapped=cache[key];cache.move_to_end(key)
        positions=np.flatnonzero(shard_ids==shard_id)
        start=0 if shard_id==0 else int(view['ends'][shard_id-1])
        local=indices[positions]-start
        for name in required:buffers[name][positions]=mapped[name][local]
        while len(cache)>32:cache.popitem(last=False)
    if not np.isfinite(buffers['features']).all() or not buffers['legal'].any(1).all():raise ValueError('Invalid prepared feature/legal batch.')
    if not np.isfinite(buffers['policy']).all() or (buffers['policy'][~buffers['legal']]!=0).any() or not np.allclose(buffers['policy'].sum(1),1,atol=1e-5,rtol=0):
        raise ValueError('Invalid prepared teacher policy batch.')
    if not np.array_equal(buffers['features'][:,16].astype(bool).reshape(len(indices),81),buffers['legal'][:,:81]):
        raise ValueError('Prepared feature and target mask disagree.')
    batch={name:torch.as_tensor(value,device=device,dtype=torch.bool if name in ['legal','score_valid','outcome_valid','ownership_valid'] else torch.float32) for name,value in buffers.items()}
    return batch

def go2_teacher_cache_summary(dataset):
    manifest=dataset['manifest']
    return [{'Split':split,'Positions':dataset[split]['count'],'Complete games':dataset[split]['game_count'],
        'Opening families':dataset[split]['family_count'],'Opponent positions':manifest['opponent_counts'][split]}
        for split in ['train','dev']]


## Validate real games and interruption recovery
The fixture compares every prepared field after an interrupted cursor is resumed in a separate Python process. It also rejects corruption, changed recipes and unsafe paths, and checks all eight spatial transforms against exact transformed game histories.

In [ ]:
def go2_teacher_array_digests(dataset):
    digests={}
    root=Path(dataset['cache_path'])
    for split in ['train','dev']:
        for field in GO2_TEACHER_FIELDS:
            digest=hashlib.sha256()
            for shard in dataset[split]['shards']:
                array=np.load(root/shard['files'][field]['path'],mmap_mode='r',allow_pickle=False)
                digest.update(array.tobytes())
            digests[split+'/'+field]=digest.hexdigest()
    return digests

def go2_validate_teacher_pipeline(project=None):
    """Real data, exact fresh-process preparation recovery, corruption and D4 checks."""
    import subprocess,sys,uuid,copy
    project=Path(project or PROJECT);suffix=uuid.uuid4().hex[:10]
    resume_id='teacher-recovery-'+suffix;full_id='teacher-uninterrupted-'+suffix
    parameters={'train_positions':256,'dev_positions':64,'max_positions_per_game':32,
        'games_per_commit':16,'seed':2026100812}
    interrupted=go2_prepare_teacher_cache(project,resume_id,**parameters,max_commits=1)
    if interrupted.get('status')!='PREPARING':raise AssertionError('Recovery fixture must stop with a committed preparation cursor.')
    before=interrupted['cursor']['commits']
    child_code="""import nbformat,json\nfrom pathlib import Path\nPROJECT=Path(%r)\nfor name in ['00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','12_teacher_dataset_and_targets.ipynb']:\n    for cell in nbformat.read(PROJECT/'experiments/ahr_v2'/name,4).cells:\n        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,name,'exec'),globals())\ndataset=go2_prepare_teacher_cache(PROJECT,%r,**%r)\nprint(json.dumps({'identity':dataset['identity'],'counts':dataset['manifest']['counts']}))\n"""%(str(project),resume_id,parameters)
    child=subprocess.run([sys.executable,'-c',child_code],cwd=project,capture_output=True,text=True,timeout=180)
    if child.returncode:raise RuntimeError('Fresh-process producer resume failed: '+child.stderr[-3000:])
    resumed=go2_teacher_dataset(project,resume_id)
    uninterrupted=go2_prepare_teacher_cache(project,full_id,**parameters)
    if go2_teacher_array_digests(resumed)!=go2_teacher_array_digests(uninterrupted):
        raise AssertionError('Fresh-process producer recovery changed feature/target/provenance arrays.')
    checks=['immutable_full_source_checksums','source_npz_checksums_and_schema','whole_history_no_suicide_psk_replay',
        'official_family_split_before_sampling','balanced_accepted_opponent_quotas','typed_external_teacher_targets',
        'masked_unavailable_or_incompatible_auxiliary_targets','fresh_process_exact_producer_resume_all_fields']
    for bad in ['../escape','with space','']:
        try:go2_teacher_cache_path(project,bad)
        except ValueError:pass
        else:raise AssertionError('Unsafe target path accepted.')
    checks.append('unsafe_cache_paths_refused')
    try:go2_prepare_teacher_cache(project,resume_id,**{**parameters,'train_positions':257})
    except ValueError:pass
    else:raise AssertionError('Mismatched configuration silently accepted.')
    checks.append('configuration_or_source_mismatch_refused')
    scratch=project/'reports/ahr_v2/teacher_data_validation'/suffix;scratch.mkdir(parents=True,exist_ok=True)
    bad_path=scratch/'value.npy';np.save(bad_path,np.array([0.25],dtype=np.float32),allow_pickle=False)
    artifact={'path':bad_path.name,'sha256':go2_file_sha(bad_path),'bytes':bad_path.stat().st_size}
    with bad_path.open('r+b') as handle:handle.seek(-1,2);handle.write(b'\xff')
    try:go2_teacher_verify_shards(scratch,[{'rows':1,'files':{'value':artifact}}])
    except ValueError:pass
    else:raise AssertionError('Corrupt prepared shard silently accepted.')
    checks.append('prepared_payload_corruption_refused')
    # Verify geometric correspondence on actual causal game history.
    source,records=go2_teacher_index(project)
    accepted=None
    for row in go2_teacher_balanced_order(records,'train')[:128]:
        converted=go2_teacher_convert_game(source['root'],row,4)
        if converted['accepted']:
            accepted=(row,converted);break
    if accepted is None:raise AssertionError('No real compatible D4 fixture found.')
    row,converted=accepted;metadata,arrays=go2_teacher_read_game(source['root'],row)
    state=go2_initial_state(9,7.5);target_ply=int(converted['arrays']['ply'][2])
    for action in arrays['actions'][:target_ply]:state=go2_step(state,int(action))
    encoded=go2_encode_state(state)
    for symmetry in range(8):
        permutation=np.array([go2_transform_action(a,9,symmetry) for a in range(81)])
        transformed=np.empty_like(encoded.reshape(21,81));transformed[:,permutation]=encoded.reshape(21,81)
        if not np.array_equal(transformed.reshape(21,9,9),go2_encode_state(go2_transform_state(state,symmetry))):
            raise AssertionError('Spatial D4 does not equal exact full-history feature transformation.')
    checks.append('all_eight_d4_full_history_feature_correspondence')
    smoke=go2_prepare_teacher_cache(project,'go9x9-teacher-smoke-v1',train_positions=1024,dev_positions=256,
        max_positions_per_game=32,games_per_commit=128)
    batch=go2_teacher_sample_batch(smoke,np.array([0,17,63,1023],dtype=np.int64))
    for name in ['features','policy','value','score','outcome','ownership']:
        if not torch.isfinite(batch[name]).all():raise AssertionError('Nonfinite real-data batch.')
    if batch['features'].shape!=(4,21,9,9) or batch['policy'].shape!=(4,82):raise AssertionError('Public batch shape mismatch.')
    if not torch.equal(batch['legal'][:,:81],batch['features'][:,16].bool().flatten(1)):raise AssertionError('Public batch legal feature mismatch.')
    checks.append('actual_real_data_public_batch_shapes_and_masks')
    report={'schema':'ahr_teacher_data_validation_v1','status':'PASSED','checks':checks,
        'fresh_process_resume':{'interrupted_commits':before,'exact_all_field_arrays':True,'cache_id':resume_id},
        'source_commit':GO2_TEACHER_COMMIT,'release_id':GO2_TEACHER_RELEASE,
        'smoke_cache_id':'go9x9-teacher-smoke-v1','smoke_identity':smoke['identity'],
        'smoke_counts':smoke['manifest']['counts'],'smoke_accepted_games':smoke['manifest']['accepted_games'],
        'smoke_rejected':smoke['manifest']['rejected'],'source_identity':go2_teacher_source_identity(project),
        'test_consumed':False,'completed_utc':go2_now()}
    go2_atomic_json(project/'reports/ahr_v2/teacher_data_validation.json',report)
    return report,smoke


In [ ]:
validation,smoke_dataset=go2_validate_teacher_pipeline(PROJECT)
import pandas as pd
display(pd.DataFrame(go2_teacher_cache_summary(smoke_dataset)))
print(f"{len(validation['checks'])} checks passed; exact fresh-process recovery verified. Test games consumed: {validation['test_consumed']}.")
print('Validation report:',PROJECT/'reports/ahr_v2/teacher_data_validation.json')

## Prepare the production bank explicitly

Run All above builds and checks only 1,024 training and 256 development samples. Notebook 15 launches the full preparation and notebook 13 training inside one detached tmux session. The explicit production preparation call is:

```python
dataset = go2_prepare_teacher_cache(
    PROJECT, cache_id='go9x9-teacher-v1',
    train_positions=500000, dev_positions=8192,
    max_positions_per_game=32, games_per_commit=128,
)
```

Repeat that exact call to resume preparation. A completed cache is immutable. To expand to a million samples, choose a **new cache ID** and registered configuration; do not change an existing run's dataset identity. Notebook 13 accepts `go2_teacher_dataset(...)` and `go2_teacher_sample_batch(...)`. It keeps teacher estimates separate from observed outcomes and never invents teacher draw probabilities from a signed scalar.

The small validation proves byte checks, legal replay, typed targets and exact recovery. It does not establish playing strength; notebook 14 measures that with independent games.